In [65]:
# Cell 1: Imports and config
import pandas as pd
import numpy as np
from scipy.stats import norm

RECOVERY = 0.40  # ISDA sovereign CDS convention
HORIZON  = 1     # 5-year CDS

DATE_COL    = 'date'
COUNTRY_COL = 'country'
DD_COL      = 'distance_to_distress'
CDS_COL     = 'cds_spread'

EXPORTERS = ['Saudi Arabia', 'Abu Dhabi', 'Qatar', 'Colombia',
             'Mexico', 'Brazil', 'Egypt', 'Malaysia']
CONTROLS  = ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa',
             'South Korea', 'Thailand', 'Turkey']

In [66]:
# Cell 2: Load M0 and compute reduced-form PD per observation
df = pd.read_csv('../output/results/M0_results_1YCDS.csv', parse_dates=[DATE_COL])
df = df[df[COUNTRY_COL].isin(EXPORTERS + CONTROLS)].copy()
df = df.sort_values([COUNTRY_COL, DATE_COL])
df = df.dropna(subset=[DD_COL, CDS_COL])

def reduced_form_pd(cds_bps, recovery=RECOVERY, T=HORIZON):
    """
    Convert observed CDS spread (bps) to risk-neutral default probability
    using standard intensity model:
        lambda = spread / (1 - RR)
        PD     = 1 - exp(-lambda * T)
    """
    spread = cds_bps / 10000
    hazard = spread / (1 - recovery)
    return 1 - np.exp(-hazard * T)

df['pd_rf'] = reduced_form_pd(df[CDS_COL])

print("Sample PD statistics:")
print(df.groupby(COUNTRY_COL)['pd_rf'].mean().sort_values().round(4))

Sample PD statistics:
country
Thailand        0.0024
South Korea     0.0029
Abu Dhabi       0.0031
Philippines     0.0031
China           0.0031
Malaysia        0.0033
Qatar           0.0038
Chile           0.0040
Indonesia       0.0051
Saudi Arabia    0.0051
Mexico          0.0070
Colombia        0.0096
Brazil          0.0128
South Africa    0.0137
Turkey          0.0374
Egypt           0.0634
Name: pd_rf, dtype: float64


In [67]:
# Cell 3: Compute alpha per country
# Solve: N(-mean_DD * alpha) = mean_PD_rf
# =>     alpha = -N^{-1}(mean_PD_rf) / mean_DD

alpha_records = []

for country in EXPORTERS + CONTROLS:
    cdf = df[df[COUNTRY_COL] == country].dropna(subset=[DD_COL, 'pd_rf'])

    if len(cdf) < 10:
        continue

    mean_dd = cdf[DD_COL].mean()
    mean_pd = cdf['pd_rf'].mean()
    mean_pd_clipped = np.clip(mean_pd, 1e-6, 1 - 1e-6)

    # Direct analytical solution
    alpha = -norm.ppf(mean_pd_clipped) / mean_dd

    alpha_records.append({
        COUNTRY_COL:  country,
        'group':      'Exporter' if country in EXPORTERS else 'Control',
        'mean_dd':    mean_dd,
        'mean_cds':   cdf[CDS_COL].mean(),
        'mean_pd_rf': mean_pd,
        'alpha':      alpha,
        'n_obs':      len(cdf),
    })

alpha_df = pd.DataFrame(alpha_records).sort_values('alpha')
print(alpha_df.to_string(index=False))

     country    group   mean_dd   mean_cds  mean_pd_rf    alpha  n_obs
   Abu Dhabi Exporter 31.886200  18.743189    0.003116 0.085782    522
    Thailand  Control 32.057768  14.338720    0.002386 0.088031    522
 Philippines  Control 26.068731  18.745183    0.003118 0.104919    522
       China  Control 24.926464  18.892579    0.003142 0.109624    522
 South Korea  Control 24.996335  17.443504    0.002902 0.110363    522
       Egypt Exporter 13.105941 403.280649    0.063429 0.116482    522
       Qatar Exporter 21.848202  23.094973    0.003838 0.122023    522
    Malaysia Exporter 21.176485  19.733100    0.003282 0.128361    522
Saudi Arabia Exporter 18.784443  30.953824    0.005137 0.136629    522
      Brazil Exporter 14.882122  77.682152    0.012839 0.149914    522
   Indonesia  Control 15.800980  30.428417    0.005054 0.162782    522
      Mexico Exporter 12.819031  41.972596    0.006965 0.191828    522
South Africa  Control  9.994803  82.748374    0.013660 0.220806    522
      

In [68]:
# Cell 4: Validate calibration
# Apply alpha and check mean calibrated PD vs mean reduced-form PD

def dd_to_implied_cds(dd_series, country_series,
                       alpha_dict,
                       recovery=RECOVERY, horizon=HORIZON):
    """
    DD_cal      = DD * alpha
    PD_cal      = N(-DD_cal)
    CDS_implied = -log(1 - PD_cal) * (1-RR) / T * 10000  [in bps]
    """
    alpha       = country_series.map(alpha_dict)
    dd_cal      = dd_series * alpha
    pd_cal      = pd.Series(norm.cdf(-dd_cal.values),
                            index=dd_series.index).clip(1e-6, 1 - 1e-6)
    cds_implied = -np.log(1 - pd_cal) * (1 - recovery) / horizon * 10000
    return cds_implied

# Build alpha dict
ALPHA_CALIBRATION = dict(zip(alpha_df[COUNTRY_COL], alpha_df['alpha']))

# Apply to M0
df['cds_implied'] = dd_to_implied_cds(df[DD_COL], df[COUNTRY_COL],
                                       ALPHA_CALIBRATION)
df['pd_cal']      = norm.cdf(-(df[DD_COL] * df[COUNTRY_COL].map(ALPHA_CALIBRATION)))

val = (
    df.groupby(COUNTRY_COL)
    .agg(
        mean_pd_cal  = ('pd_cal',      'mean'),
        mean_pd_rf   = ('pd_rf',       'mean'),
        mean_cds_obs = (CDS_COL,       'mean'),
        mean_cds_imp = ('cds_implied', 'mean'),
    )
    .reset_index()
)
val['pd_diff']  = (val['mean_pd_cal']  - val['mean_pd_rf']).round(4)
val['cds_diff'] = (val['mean_cds_imp'] - val['mean_cds_obs']).round(2)

print("\nValidation: Mean calibrated PD vs reduced-form PD")
print("=" * 60)
print(val.to_string(index=False))


Validation: Mean calibrated PD vs reduced-form PD
     country  mean_pd_cal  mean_pd_rf  mean_cds_obs  mean_cds_imp  pd_diff  cds_diff
   Abu Dhabi     0.027196    0.003116     18.743189    171.643486   0.0241    152.90
      Brazil     0.021444    0.012839     77.682152    131.448964   0.0086     53.77
       Chile     0.009601    0.004003     24.083251     58.429120   0.0056     34.35
       China     0.010909    0.003142     18.892579     66.488426   0.0078     47.60
    Colombia     0.017768    0.009576     57.819811    108.401822   0.0082     50.58
       Egypt     0.123463    0.063429    403.280649    864.282163   0.0600    461.00
   Indonesia     0.018379    0.005054     30.428417    114.386717   0.0133     83.96
    Malaysia     0.008084    0.003282     19.733100     49.055594   0.0048     29.32
      Mexico     0.016920    0.006965     41.972596    103.888422   0.0100     61.92
 Philippines     0.018627    0.003118     18.745183    115.157023   0.0155     96.41
       Qatar  

In [69]:
print("ALPHA_CALIBRATION = {")
for _, row in alpha_df.iterrows():
    print(f"    '{row[COUNTRY_COL]}': {row['alpha']:.6f},")
print("}")

ALPHA_CALIBRATION = {
    'Abu Dhabi': 0.085782,
    'Thailand': 0.088031,
    'Philippines': 0.104919,
    'China': 0.109624,
    'South Korea': 0.110363,
    'Egypt': 0.116482,
    'Qatar': 0.122023,
    'Malaysia': 0.128361,
    'Saudi Arabia': 0.136629,
    'Brazil': 0.149914,
    'Indonesia': 0.162782,
    'Mexico': 0.191828,
    'South Africa': 0.220806,
    'Turkey': 0.226695,
    'Colombia': 0.237888,
    'Chile': 0.322843,
}
